<a href="https://colab.research.google.com/github/Towa-1103/Experiment/blob/main/res01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 1. 必要なライブラリのみをインストール（torchは除外）
!pip install -q -U transformers accelerate sentence-transformers


# 2. LLM（Qwen-2.5-3B-Instruct）の読み込み
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen2.5-3B-Instruct"
print(f"[{model_id}] の読み込みを開始します...")

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,  # メモリ節約（最新仕様に対応）
    device_map="auto",  # GPUへ自動配置
)

# 正常に読み込めたかの確認
print("\n✅ セットアップ完了")
print(f"・使用デバイス: {model.device}")
if torch.cuda.is_available():
  print(f"・GPU名: {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 93.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 49.6 MB/s eta 0:00:00
[Qwen/Qwen2.5-3B-Instruct] の読み込みを開始します...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


✅ セットアップ完了
・使用デバイス: cuda:0
・GPU名: Tesla T4


In [5]:
import os

# 必ず /content を基準にする
%cd /content

REPO_URL = "https://github.com/Towa-1103/Experiment.git"
TARGET_DIR = "/content/Experiment"

if not os.path.exists(TARGET_DIR):
  !git clone {REPO_URL}
  %cd {TARGET_DIR}
else:
  %cd {TARGET_DIR}
  !git pull

print("\n 現在地:", os.getcwd())
print("\n リポジトリの同期が完了しました。")

/content
/content/Experiment
Already up to date.

 現在地: /content/Experiment

 リポジトリの同期が完了しました。


In [ ]:
!git pull
import importlib
import dialogue_processor
import memory_manager

importlib.reload(dialoge_processor)
importlib.reload(memory_manager)

from dialogue_processor import extract_memories_from_log
from memory_manager import MemoryManager

# -----入力フォーム-----
LOG_FILE = ".Friend_A_2025.txt"
#-----------------------

if not os.path.exists(LOG_FILE):
  raise FileNotFoundError(
      f"'{LOG_FILE}' が見つかりません。ファイル名を確認してください。"
  )

manager = MemoryManager("memories.json")
print(f"処理前の総記憶数: {len(manager.get_all())}件")

with open(LOG_FILE, "r", encoding="utf-8") as f:
  raw_log = f.read()

print("\n会話ログから記憶抽出中...")
extracted = extract_memories_from_log(
    raw_log, tokenizer, model, min_importance=3
)
print(f"抽出された候補: {len(extracted)}　件")

added_count = manager.add_memories(extracted)

print("\n" + "=" * 40)
print(f"新規追記: {added_count}件")
print(f"処理後の総記憶数: {len(manager.get_all())}件\n")

#保存されたmemories.jsonの確認
!cat memories.json